# 03 · Evaluation

Scores the trained model on a **separate test zip** (different room or day). The model receives
the image and the meta vector only; LiDAR depth is used solely as the truth to score against.

Compared here:
- the model with the focus input,
- the image-only ablation,
- a no-learning baseline that takes the calibrated focus distance as the object's distance.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "python"))  # use the package without installing it
DATA = Path("../data")      # session zips exported from the phone
MODELS = Path("../models")  # checkpoints, calibration.json, UNetUltra.mlpackage
MODELS.mkdir(exist_ok=True)

In [ ]:
from unetultra.calibration import LensCalibration
from unetultra.dataset import SessionDataset
from unetultra.session import find_sessions
from unetultra.train import load_model

TEST_ZIPS = find_sessions(DATA, "test")        # only sessions marked split: test
assert TEST_ZIPS, "no test session found in ../data"
calibration = LensCalibration.load(MODELS / "calibration.json")
model, checkpoint = load_model(MODELS / "model.pt")
ablation, _ = load_model(MODELS / "model_image_only.pt")
test_set = SessionDataset(TEST_ZIPS, calibration)
test_set_image_only = SessionDataset(TEST_ZIPS, calibration, use_meta=False)
print(f"{len(test_set)} test samples from {len(TEST_ZIPS)} session(s); checkpoint from epoch {checkpoint['epoch']}")

## Metrics

In [ ]:
from unetultra import metrics

rows, image_rows, predictions = metrics.evaluate_model(model, test_set)
rows_ablation, image_rows_ablation, _ = metrics.evaluate_model(ablation, test_set_image_only)
summary = metrics.summarise(rows, image_rows)
summary_ablation = metrics.summarise(rows_ablation, image_rows_ablation)
print(metrics.format_summary({"with focus": summary, "image only": summary_ablation}))

### Focus-only baseline

Using the calibrated focus distance as the distance of the object the lens focused on, with no
model at all. If the model cannot beat this, it has learned nothing about distance beyond focus.

In [ ]:
baseline, on_targets = summary["distance_focus_baseline"], summary["distance_model_on_focus_targets"]
print(f"focused objects: {baseline['n']}")
print(f"  focus distance alone : {baseline['mae']:.2f} cm mean abs error ({baseline['mape']:.1f}%)")
print(f"  model with focus     : {on_targets['mae']:.2f} cm mean abs error ({on_targets['mape']:.1f}%)")

### Size: before and after the bias fix

The reference method (2nd–98th percentile between pixel centres) under-reports size by about 4%
and is thrown off by depth outliers at the object's edge. The fix measures outlier pixels at
the object's median depth and uses the full pixel footprint.

In [ ]:
print(metrics.format_size_fix(summary))
lidar_vs_ruler = summary["lidar_width_vs_measured"]
if lidar_vs_ruler["n"]:
    print(f"\nFor reference, LiDAR-derived width against the ruler: {lidar_vs_ruler['mae']:.2f} cm "
          f"({lidar_vs_ruler['mape']:.1f}%) over {lidar_vs_ruler['n']} objects")

### By distance

In [ ]:
for name, r in (("with focus", rows), ("image only", rows_ablation)):
    print(name)
    for row in metrics.distance_error_by_bin(r):
        print(f"  {row['bin_cm'][0]:>3}-{row['bin_cm'][1]:<3} cm  n={row['n']:<3} "
              f"mean abs {row['mae']:.2f} cm ({row['mape']:.1f}%)  bias {row['bias']:+.2f} cm")

## Plots

In [ ]:
from unetultra import plots

results = {"with focus": rows, "image only": rows_ablation}
plots.predicted_vs_true_distance(results)
plots.error_vs_distance(results)
plots.error_vs_distance({"with focus": rows}, key="pred_width_cm", truth="true_width_cm", ylabel="width error (cm)")
plots.error_vs_lens(rows);

In [ ]:
plots.example_overlays(test_set, predictions, rows);

## Verdict

Generated from the numbers above. Add your own reading underneath after looking at the plots.

In [ ]:
print(metrics.verdict(summary, summary_ablation, rows))

_Your notes:_